# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR^2 dataset using the `mlcroissant` library and references all entities using their `@id`s.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata
print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

```python
record_sets = dataset.metadata.record_sets # List of record set objects
fields = dataset.metadata.fields # List of field objects
```
Below, we print the available record sets and their fields using their `@id`s:

In [ ]:
# Print available record sets and their fields with @id references
record_sets = getattr(dataset.metadata, 'record_sets', [])
fields = getattr(dataset.metadata, 'fields', [])

print("Record Sets:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs['@id']}  (name: {rs.get('name', '')})")
    rs_fields = rs.get('fields', [])
    for field in rs_fields:
        print(f"    - Field @id: {field['@id']}  (name: {field.get('name', '')})")
print("\nAll fields in dataset:")
for f in fields:
    print(f"- Field @id: {f['@id']}  (name: {f.get('name', '')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview above.

In [ ]:
# Find all record set @id's
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load each record set into a dataframe indexed by @id
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Print columns of the first record set
if dataframes:
    first_rs_id = list(dataframes.keys())[0]
    print(f"Record set columns (@id) for '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()
else:
    print('No record set data found.')

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

You must reference specific fields using their `@id`s.

In [ ]:
# Example: Select a numeric field for analysis using its @id
# Replace these with actual @ids from the printed overview above
if dataframes:
    # Use the first record set and field for example
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]

    # Attempt to select a numeric field automatically
    numeric_field_id = None
    for col in df.columns:
        # Try guessing if numeric by dtype
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

    if numeric_field_id:
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].notnull().any() else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try grouping by another field (categorical)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print('No grouping field found for grouping analysis.')
    else:
        print('No numeric fields found in the record set.')
else:
    print('No dataframes available for EDA.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset (referencing field `@id`s).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of field (@id): {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    
    # Optionally, show relationship between group_field_id and numeric_field_id
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f'{numeric_field_id} by {group_field_id} (@id)')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset was successfully loaded and entities referenced via their `@id`s.
- Record sets, fields, and columns were identified using their global IDs.
- Filtered, normalized, and grouped data using numeric and categorical fields.
- Visualization highlighted data distributions and relationships.

This notebook can serve as a reproducible template for FAIR data exploration using `mlcroissant` and referencing entities by their `@id`.